# Meteorite Image Classification (timm + wandb)

This notebook trains an image classifier for the meteorite task using `timm.efficientnetv2_s`, tracks experiments with Weights & Biases, and exports `submission.csv`.

In [ ]:
# Optional dependency bootstrap
import importlib
import subprocess
import sys

def ensure_packages(requirements):
    missing_pip_packages = []
    for import_name, pip_name in requirements.items():
        try:
            importlib.import_module(import_name)
        except ModuleNotFoundError:
            missing_pip_packages.append(pip_name)
    if missing_pip_packages:
        print(f'Installing missing packages: {missing_pip_packages}')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing_pip_packages])

ensure_packages({
    'timm': 'timm',
    'wandb': 'wandb',
    'optuna': 'optuna',
    'sklearn': 'scikit-learn',
    'albumentations': 'albumentations',
    'cv2': 'opencv-python-headless',
    'safetensors': 'safetensors',
})

In [ ]:
import os
import random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, UnidentifiedImageError
from tqdm.auto import tqdm
from IPython.display import display
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler

import albumentations as A
from albumentations.pytorch import ToTensorV2

import timm
import wandb
import optuna

from dataset import StoneDataset, resolve_train_paths, resolve_test_paths

In [ ]:
# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
# Config
CFG = {
    'model_name': 'tf_efficientnetv2_s',
    'image_size': 224,
    'num_classes': 2,
    'batch_size': 32,
    'num_epochs': 16,
    'head_only_epochs': 3,
    'unfreeze_last_n_blocks': 4,
    'learning_rate': 3e-4,
    'head_lr': 6e-4,
    'backbone_lr': 1.2e-5,
    'weight_decay': 0.05,
    'num_workers': 2,
    'val_ratio': 0.2,
    'project_name': 'sta326-meteorite',
    'run_name': 'tf_efficientnetv2_s-two-stage',
    'output_dir': './logs',
    # Pretraining options
    'use_pretrained': True,
    'use_local_pretrained': True,
    'local_weight_path': './weights/model.safetensors',
    # Regularization in classification head
    'head_dropout': 0.5,
    'label_smoothing': 0.2,
    # Loss setup
    'use_focal_loss': True,
    'focal_gamma': 2.0,
    # If None, alpha is estimated from train split label prior each run.
    'focal_alpha': None,
    'focal_alpha_smoothing': 1e-6,
    # Mixup/Cutmix strategy: keep mixup mild, disable cutmix by default.
    'mixup_alpha': 0.2,
    'mixup_prob': 0.5,
    'cutmix_alpha': 0.0,
    # Test-time augmentation options
    'use_tta': True,
    'tta_mode': 'simple_flip',
    'tta_views': 4,
    # Test prior: non-meteorite : meteorite = 4 : 1
    'test_class_prior': [0.8, 0.2],
    'bayes_threshold_trials': 40,
    'default_threshold': 0.5,
    # Data quality and balancing options
    'clean_index_csv': './logs/clean_train_ids.csv',
    'manual_clean_csv': './logs/manual_clean_labels.csv',
    'audit_csv': './logs/train_audit.csv',
    'min_image_side': 64,
    'max_image_side': 4096,
    'audit_preview_per_class': 10,
    # Keep only one imbalance strategy by default: class-weighted loss.
    'use_weighted_sampler': False,
    'use_manual_sample_weight': True,
    'use_class_weighted_loss': True,
    # If enabled, enforce test-time positive ratio close to test prior.
    'use_prior_threshold_on_test': True,
    # Constrained threshold search options
    'threshold_scan_min': 0.05,
    'threshold_scan_max': 0.95,
    'threshold_scan_step': 0.005,
    'test_pos_rate_min': 0.18,
    'test_pos_rate_max': 0.24,
    'test_pos_rate_target': 0.20,
    'threshold_f1_tolerance': 0.003,
    'threshold_fallback_f1_penalty_weight': 10.0,
}

os.makedirs(CFG['output_dir'], exist_ok=True)

DATA_ROOT_CANDIDATES = [
    './dataset',
    '/data/meteorite-identification',
    './data',
]

def detect_data_root(candidates):
    for root in candidates:
        try:
            resolve_train_paths(root)
            resolve_test_paths(root)
            return root
        except Exception:
            continue
    raise FileNotFoundError(
        'No valid dataset root found. Expected train_images/train_labels.csv/test_images/sample_submission.csv'
    )

DATA_ROOT = detect_data_root(DATA_ROOT_CANDIDATES)
print('DATA_ROOT =', DATA_ROOT)

In [ ]:
# Albumentations adapter for StoneDataset(PIL -> tensor)
class AlbumentationsTransform:
    def __init__(self, aug):
        self.aug = aug

    def __call__(self, image):
        image_np = np.array(image)
        augmented = self.aug(image=image_np)
        return augmented['image']


train_transform = AlbumentationsTransform(
    A.Compose([
        A.Resize(CFG['image_size'], CFG['image_size']),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Rotate(limit=180, p=0.8, border_mode=cv2.BORDER_REFLECT_101),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05, p=0.7),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])
)

eval_transform = AlbumentationsTransform(
    A.Compose([
        A.Resize(CFG['image_size'], CFG['image_size']),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])
)

In [ ]:
# Build datasets and loaders with data audit + clean index + weighted sampler
full_plain_dataset = StoneDataset(root=DATA_ROOT, split='train', transforms=None)

def audit_and_save_clean_index(dataset_obj, cfg):
    records = []
    for idx, (img_path, label) in enumerate(zip(dataset_obj.samples, dataset_obj.labels)):
        row = {
            'orig_index': idx,
            'id': os.path.basename(img_path),
            'path': img_path,
            'label': int(label),
            'status': 'keep',
            'reason': '',
            'width': None,
            'height': None,
        }
        try:
            with Image.open(img_path) as img:
                img.verify()
            with Image.open(img_path) as img:
                w, h = img.size
            row['width'], row['height'] = int(w), int(h)
            if min(w, h) < cfg['min_image_side']:
                row['status'] = 'drop'
                row['reason'] = f"too_small(<{cfg['min_image_side']})"
            elif max(w, h) > cfg['max_image_side']:
                row['status'] = 'drop'
                row['reason'] = f"too_large(>{cfg['max_image_side']})"
        except (FileNotFoundError, UnidentifiedImageError, OSError, ValueError) as e:
            row['status'] = 'drop'
            row['reason'] = f'read_error:{type(e).__name__}'
        records.append(row)

    audit_df = pd.DataFrame(records)
    audit_df.to_csv(cfg['audit_csv'], index=False)

    clean_df = audit_df[audit_df['status'] == 'keep'].copy()
    clean_df.to_csv(cfg['clean_index_csv'], index=False)

    print('Audit report saved:', cfg['audit_csv'])
    print('Clean index saved:', cfg['clean_index_csv'])
    print('Total train images:', len(audit_df))
    print('Kept images:', len(clean_df))
    print('Dropped images:', len(audit_df) - len(clean_df))
    print('Drop reasons:')
    print(audit_df['reason'].replace('', 'kept').value_counts().head(10))

    if len(clean_df):
        print('Clean label distribution:')
        print(clean_df['label'].value_counts().sort_index())
        print('Image size summary (clean):')
        print(clean_df[['width', 'height']].describe().round(2))

    clean_indices_local = clean_df['orig_index'].astype(int).tolist()
    return clean_indices_local, audit_df, clean_df

clean_indices, audit_df, clean_df = audit_and_save_clean_index(full_plain_dataset, CFG)
if len(clean_indices) == 0:
    raise RuntimeError('No clean training images available after audit.')

manual_clean_path = CFG.get('manual_clean_csv', '').strip()
manual_decision_df = None
train_relabel_map = {}
train_weight_map = {}
if manual_clean_path and os.path.exists(manual_clean_path):
    manual_decision_df = pd.read_csv(manual_clean_path)

    if len(manual_decision_df) == 0:
        manual_decision_df = None
        print('skip manual cleaning')
    else:
        required_cols = {'id', 'sample_status'}
        if not required_cols.issubset(manual_decision_df.columns):
            raise ValueError(f"{manual_clean_path} must contain columns: {sorted(required_cols)}")

        manual_decision_df['id'] = manual_decision_df['id'].astype(str)
        manual_decision_df['sample_status'] = manual_decision_df['sample_status'].astype(str).str.lower().str.strip()
        if 'sample_weight' not in manual_decision_df.columns:
            manual_decision_df['sample_weight'] = np.nan
        if 'final_label' not in manual_decision_df.columns:
            manual_decision_df['final_label'] = np.nan

        status_default_weight = {'clean': 1.0, 'hard': 1.3, 'drop': 0.0, 'relabel': 1.0}
        valid_status = set(status_default_weight.keys())
        bad_status = manual_decision_df[~manual_decision_df['sample_status'].isin(valid_status)]
        if len(bad_status):
            raise ValueError(f"Invalid sample_status values in {manual_clean_path}: {sorted(bad_status['sample_status'].unique())}")

        decision_by_id = {row['id']: row for _, row in manual_decision_df.iterrows()}
        filtered_clean_indices = []
        dropped_count = 0
        relabeled_count = 0

        for orig_idx in clean_indices:
            img_id = os.path.basename(full_plain_dataset.samples[int(orig_idx)])
            decision = decision_by_id.get(img_id)
            if decision is None:
                train_weight_map[int(orig_idx)] = 1.0
                filtered_clean_indices.append(int(orig_idx))
                continue

            status = str(decision['sample_status']).lower().strip()
            if status == 'drop':
                dropped_count += 1
                continue

            final_label = pd.to_numeric(decision.get('final_label', np.nan), errors='coerce')
            if status == 'relabel' and not np.isnan(final_label):
                train_relabel_map[int(orig_idx)] = int(final_label)
                relabeled_count += 1

            sample_weight = pd.to_numeric(decision.get('sample_weight', np.nan), errors='coerce')
            if np.isnan(sample_weight):
                sample_weight = status_default_weight.get(status, 1.0)
            train_weight_map[int(orig_idx)] = float(sample_weight)
            filtered_clean_indices.append(int(orig_idx))

        clean_indices = np.array(filtered_clean_indices, dtype=np.int64)
        print('Manual clean file loaded:', manual_clean_path)
        print('Manual decisions rows:', len(manual_decision_df))
        print('Dropped by manual clean:', dropped_count)
        print('Relabeled for training:', relabeled_count)
else:
    clean_indices = np.array(clean_indices, dtype=np.int64)

if manual_decision_df is None:
    clean_indices = np.array(clean_indices, dtype=np.int64)
    for orig_idx in clean_indices.tolist():
        train_weight_map[int(orig_idx)] = 1.0

split_labels = np.array([int(full_plain_dataset.labels[i]) for i in clean_indices], dtype=np.int64)
if len(np.unique(split_labels)) >= 2:
    train_indices_np, val_indices_np = train_test_split(
        clean_indices,
        test_size=CFG['val_ratio'],
        random_state=SEED,
        shuffle=True,
        stratify=split_labels,
    )
else:
    rng = np.random.default_rng(SEED)
    shuffled_indices = clean_indices.copy()
    rng.shuffle(shuffled_indices)
    num_total = len(shuffled_indices)
    num_val = int(num_total * CFG['val_ratio'])
    num_train = num_total - num_val
    train_indices_np = shuffled_indices[:num_train]
    val_indices_np = shuffled_indices[num_train:]

train_indices = np.array(train_indices_np, dtype=np.int64).tolist()
val_indices = np.array(val_indices_np, dtype=np.int64).tolist()

num_total = len(clean_indices)
num_train = len(train_indices)
num_val = len(val_indices)
overall_pos_rate = float(split_labels.mean()) if len(split_labels) else 0.0
train_pos_rate = float(np.mean([full_plain_dataset.labels[i] for i in train_indices])) if num_train else 0.0
val_pos_rate = float(np.mean([full_plain_dataset.labels[i] for i in val_indices])) if num_val else 0.0
print('Split summary (overall/train/val sizes):', num_total, num_train, num_val)
print('Positive-rate summary (overall/train/val):', round(overall_pos_rate, 4), round(train_pos_rate, 4), round(val_pos_rate, 4))

train_dataset_raw = StoneDataset(root=DATA_ROOT, split='train', transforms=train_transform)
val_dataset_raw = StoneDataset(root=DATA_ROOT, split='train', transforms=eval_transform)
test_dataset = StoneDataset(root=DATA_ROOT, split='test', transforms=eval_transform)

# Apply manual relabel decisions only to training subset labels.
for idx, new_label in train_relabel_map.items():
    if idx in train_indices:
        train_dataset_raw.labels[idx] = int(new_label)

train_dataset = Subset(train_dataset_raw, train_indices)
val_dataset = Subset(val_dataset_raw, val_indices)

train_sampler = None
if (CFG['use_weighted_sampler'] or CFG.get('use_manual_sample_weight', False)) and len(train_indices) > 0:
    train_labels_local = np.array([train_dataset_raw.labels[i] for i in train_indices], dtype=np.int64)

    sample_weights = np.ones(len(train_indices), dtype=np.float64)
    class_counts = np.bincount(train_labels_local, minlength=CFG['num_classes'])
    if CFG['use_weighted_sampler']:
        class_sample_weights = np.zeros(CFG['num_classes'], dtype=np.float64)
        nonzero = class_counts > 0
        class_sample_weights[nonzero] = 1.0 / class_counts[nonzero]
        sample_weights = sample_weights * class_sample_weights[train_labels_local]
    else:
        class_sample_weights = np.ones(CFG['num_classes'], dtype=np.float64)

    if CFG.get('use_manual_sample_weight', False):
        manual_weights_local = np.array([train_weight_map.get(i, 1.0) for i in train_indices], dtype=np.float64)
        sample_weights = sample_weights * manual_weights_local

    train_sampler = WeightedRandomSampler(
        weights=torch.as_tensor(sample_weights, dtype=torch.double),
        num_samples=len(sample_weights),
        replacement=True,
    )
    print('WeightedRandomSampler enabled.')
    print('Train class counts:', class_counts.tolist())
    print('Train class sample weights:', class_sample_weights.round(6).tolist())
    if CFG.get('use_manual_sample_weight', False):
        print('Manual sample weight stats:', {
            'min': float(np.min(manual_weights_local)),
            'max': float(np.max(manual_weights_local)),
            'mean': float(np.mean(manual_weights_local)),
        })

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG['batch_size'],
    shuffle=(train_sampler is None),
    sampler=train_sampler,
    num_workers=CFG['num_workers'],
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CFG['batch_size'],
    shuffle=False,
    num_workers=CFG['num_workers'],
    pin_memory=torch.cuda.is_available(),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CFG['batch_size'],
    shuffle=False,
    num_workers=CFG['num_workers'],
    pin_memory=torch.cuda.is_available(),
)

print(f'Train/Val/Test sizes: {len(train_dataset)}/{len(val_dataset)}/{len(test_dataset)}')

In [ ]:
# Data quality visualization + loader sanity check
def show_random_by_label(clean_df_local, n_per_class=10):
    label_to_name = {0: 'non-meteorite', 1: 'meteorite'}
    fig, axes = plt.subplots(2, n_per_class, figsize=(2.2 * n_per_class, 5))
    for row_idx, label in enumerate([0, 1]):
        subset = clean_df_local[clean_df_local['label'] == label]
        if len(subset) == 0:
            for col in range(n_per_class):
                axes[row_idx, col].axis('off')
            continue
        sample_df = subset.sample(n=min(n_per_class, len(subset)), random_state=SEED)
        for col in range(n_per_class):
            ax = axes[row_idx, col]
            if col < len(sample_df):
                path = sample_df.iloc[col]['path']
                with Image.open(path) as img:
                    ax.imshow(img.convert('RGB'))
                ax.set_title(label_to_name[label], fontsize=9)
            ax.axis('off')
    plt.tight_layout()
    plt.show()

show_random_by_label(clean_df, n_per_class=CFG['audit_preview_per_class'])

images, labels = next(iter(train_loader))
print('Batch image shape:', images.shape)
print('Batch label shape:', labels.shape)
print('Label min/max:', labels.min().item(), labels.max().item())

In [ ]:
# Build model
def load_local_weights_flexible(model, weight_path):
    if not os.path.exists(weight_path):
        raise FileNotFoundError(f'Local weight file not found: {weight_path}')

    if weight_path.endswith('.safetensors'):
        from safetensors.torch import load_file as safe_load_file
        raw_state = safe_load_file(weight_path)
    else:
        ckpt = torch.load(weight_path, map_location='cpu')
        if isinstance(ckpt, dict) and 'state_dict' in ckpt:
            raw_state = ckpt['state_dict']
        else:
            raw_state = ckpt

    model_state = model.state_dict()
    filtered_state = {}
    skipped_keys = []

    for k, v in raw_state.items():
        key = k[7:] if k.startswith('module.') else k
        if key in model_state and model_state[key].shape == v.shape:
            filtered_state[key] = v
        else:
            skipped_keys.append(key)

    missing, unexpected = model.load_state_dict(filtered_state, strict=False)
    return {
        'loaded_tensors': len(filtered_state),
        'skipped_tensors': len(skipped_keys),
        'missing_keys': len(missing),
        'unexpected_keys': len(unexpected),
    }

pretrained_enabled = False
loaded_from = 'random_init'
weight_path = CFG.get('local_weight_path', '').strip()

if CFG.get('use_local_pretrained', False) and weight_path:
    model = timm.create_model(
        CFG['model_name'],
        pretrained=False,
        num_classes=CFG['num_classes'],
        drop_rate=CFG['head_dropout'],
    ).to(device)
    try:
        load_stats = load_local_weights_flexible(model, weight_path)
        pretrained_enabled = load_stats['loaded_tensors'] > 0
        loaded_from = f'local:{weight_path}' if pretrained_enabled else 'random_init'
        print('Loaded local weights stats:', load_stats)
    except Exception as e:
        print(f'Local weight load failed: {e}')
        print('Fallback to timm pretrained/random strategy.')
        if CFG['use_pretrained']:
            try:
                model = timm.create_model(
                    CFG['model_name'],
                    pretrained=True,
                    num_classes=CFG['num_classes'],
                    drop_rate=CFG['head_dropout'],
                ).to(device)
                pretrained_enabled = True
                loaded_from = 'timm_pretrained'
            except RuntimeError as e2:
                print(f'Timm pretrained load failed: {e2}')
                model = timm.create_model(
                    CFG['model_name'],
                    pretrained=False,
                    num_classes=CFG['num_classes'],
                    drop_rate=CFG['head_dropout'],
                ).to(device)
                pretrained_enabled = False
                loaded_from = 'random_init'
        else:
            model = timm.create_model(
                CFG['model_name'],
                pretrained=False,
                num_classes=CFG['num_classes'],
                drop_rate=CFG['head_dropout'],
            ).to(device)
            pretrained_enabled = False
            loaded_from = 'random_init'
else:
    if CFG['use_pretrained']:
        try:
            model = timm.create_model(
                CFG['model_name'],
                pretrained=True,
                num_classes=CFG['num_classes'],
                drop_rate=CFG['head_dropout'],
            ).to(device)
            pretrained_enabled = True
            loaded_from = 'timm_pretrained'
        except RuntimeError as e:
            print(f'Pretrained load failed for {CFG["model_name"]}: {e}')
            print('Fallback to pretrained=False for this timm version/model.')
            model = timm.create_model(
                CFG['model_name'],
                pretrained=False,
                num_classes=CFG['num_classes'],
                drop_rate=CFG['head_dropout'],
            ).to(device)
            pretrained_enabled = False
            loaded_from = 'random_init'
    else:
        model = timm.create_model(
            CFG['model_name'],
            pretrained=False,
            num_classes=CFG['num_classes'],
            drop_rate=CFG['head_dropout'],
        ).to(device)
        pretrained_enabled = False
        loaded_from = 'random_init'

class FocalLoss(nn.Module):
    """Numerically stable focal loss for 2-class logits."""

    def __init__(self, gamma=2.0, alpha=None, reduction='mean'):
        super().__init__()
        self.gamma = float(gamma)
        if reduction not in ('none', 'mean', 'sum'):
            raise ValueError(f'Invalid reduction: {reduction}')
        self.reduction = reduction
        self.alpha = None if alpha is None else torch.as_tensor(alpha, dtype=torch.float32)

    def forward(self, logits, targets):
        if logits.ndim != 2 or logits.size(1) != 2:
            raise ValueError(f'FocalLoss expects logits shape [N, 2], got {tuple(logits.shape)}')

        targets = targets.view(-1).long()
        if targets.size(0) != logits.size(0):
            raise ValueError(
                f'Mismatched batch size between logits ({logits.size(0)}) and targets ({targets.size(0)})'
            )

        log_probs = F.log_softmax(logits, dim=1)
        log_pt = log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        pt = log_pt.exp()

        focal_factor = (1.0 - pt).clamp(min=0.0).pow(self.gamma)
        loss = -focal_factor * log_pt

        if self.alpha is not None:
            alpha = self.alpha.to(device=logits.device, dtype=logits.dtype).view(-1)
            if alpha.numel() != 2:
                raise ValueError(f'alpha must have length 2 for binary classification, got {alpha.numel()}')
            loss = alpha.gather(0, targets) * loss

        if self.reduction == 'mean':
            return loss.mean()
        if self.reduction == 'sum':
            return loss.sum()
        return loss


if CFG['num_classes'] != 2:
    raise ValueError('This notebook focal setup currently supports binary classification only (num_classes=2).')

# Estimate class prior from the current train split for loss weighting.
train_labels_for_loss = np.array([train_dataset_raw.labels[i] for i in train_indices], dtype=np.int64)
train_label_counts = np.bincount(train_labels_for_loss, minlength=CFG['num_classes']).astype(np.float64)
alpha_smoothing = float(CFG.get('focal_alpha_smoothing', 1e-6))
train_label_prior = (train_label_counts + alpha_smoothing) / (train_label_counts.sum() + alpha_smoothing * CFG['num_classes'])
class_weights_np = 1.0 / train_label_prior
class_weights_np = class_weights_np / class_weights_np.mean()
class_weights = torch.tensor(class_weights_np, dtype=torch.float32, device=device)

if CFG.get('focal_alpha') is None:
    focal_alpha = class_weights.detach().clone()
else:
    focal_alpha = torch.as_tensor(CFG['focal_alpha'], dtype=torch.float32, device=device).view(-1)
    if focal_alpha.numel() != CFG['num_classes']:
        raise ValueError(
            f"CFG['focal_alpha'] must have length {CFG['num_classes']}, got {focal_alpha.numel()}"
        )

if CFG.get('use_focal_loss', False):
    criterion = FocalLoss(gamma=CFG['focal_gamma'], alpha=focal_alpha, reduction='mean')
    loss_name = 'focal'
else:
    if CFG['use_class_weighted_loss']:
        criterion = nn.CrossEntropyLoss(
            weight=class_weights,
            label_smoothing=CFG['label_smoothing'],
        )
    else:
        criterion = nn.CrossEntropyLoss(label_smoothing=CFG['label_smoothing'])
    loss_name = 'cross_entropy'

print('Model ready:', CFG['model_name'])
print('Using pretrained backbone:', pretrained_enabled)
print('Pretrained source:', loaded_from)
print('Head dropout:', CFG['head_dropout'])
print('Label smoothing:', CFG['label_smoothing'])
print('Class prior (train split):', np.round(train_label_prior, 4).tolist())
print('Train label counts:', train_label_counts.astype(int).tolist())
print('Class prior (test):', CFG['test_class_prior'])
print('Use class-weighted loss:', CFG['use_class_weighted_loss'])
print('Class weights used:', class_weights.detach().cpu().numpy().round(4).tolist())
print('Loss type:', loss_name)
if loss_name == 'focal':
    print('Focal gamma:', CFG['focal_gamma'])
    print('Focal alpha used:', focal_alpha.detach().cpu().numpy().round(4).tolist())

In [ ]:
def _sample_mixup_data(images, labels, alpha=0.2):
    if alpha <= 0:
        return images, labels, labels, 1.0
    lam = np.random.beta(alpha, alpha)
    index = torch.randperm(images.size(0), device=images.device)
    mixed_images = lam * images + (1 - lam) * images[index, :]
    labels_a, labels_b = labels, labels[index]
    return mixed_images, labels_a, labels_b, lam


def _mixup_loss(criterion, logits, labels_a, labels_b, lam):
    return lam * criterion(logits, labels_a) + (1 - lam) * criterion(logits, labels_b)


def _tta_forward_probs(model, images, mode='simple_flip', views=4):
    if mode != 'simple_flip' or views <= 1:
        logits = model(images)
        return torch.softmax(logits, dim=1)[:, 1]

    logits_list = []
    logits_list.append(model(images))

    if views >= 2:
        logits_list.append(model(torch.flip(images, dims=[3])))
    if views >= 3:
        logits_list.append(model(torch.flip(images, dims=[2])))
    if views >= 4:
        logits_list.append(model(torch.flip(images, dims=[2, 3])))

    prob_stack = [torch.softmax(lg, dim=1)[:, 1] for lg in logits_list]
    return torch.stack(prob_stack, dim=0).mean(dim=0)


def set_trainable_layers(model, unfreeze_last_n_blocks=0):
    for param in model.parameters():
        param.requires_grad = False

    # Always train classification head + final conv/bn around head.
    head_keywords = ('classifier', 'head', 'fc', 'conv_head', 'bn2')
    for name, param in model.named_parameters():
        if any(k in name for k in head_keywords):
            param.requires_grad = True

    # Unfreeze last N feature blocks for stage-2 finetuning.
    if unfreeze_last_n_blocks > 0:
        block_ids = []
        for name, _ in model.named_parameters():
            if name.startswith('blocks.'):
                parts = name.split('.')
                if len(parts) > 1 and parts[1].isdigit():
                    block_ids.append(int(parts[1]))
        if block_ids:
            max_block_id = max(block_ids)
            min_keep_id = max(0, max_block_id - unfreeze_last_n_blocks + 1)
            for name, param in model.named_parameters():
                if name.startswith('blocks.'):
                    parts = name.split('.')
                    if len(parts) > 1 and parts[1].isdigit() and int(parts[1]) >= min_keep_id:
                        param.requires_grad = True


def build_optimizer(model, cfg):
    head_keywords = ('classifier', 'head', 'fc', 'conv_head', 'bn2')
    head_params, backbone_params = [], []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if any(k in name for k in head_keywords):
            head_params.append(param)
        else:
            backbone_params.append(param)

    param_groups = []
    if backbone_params:
        param_groups.append({
            'params': backbone_params,
            'lr': cfg['backbone_lr'],
            'weight_decay': cfg['weight_decay'],
        })
    if head_params:
        param_groups.append({
            'params': head_params,
            'lr': cfg['head_lr'],
            'weight_decay': cfg['weight_decay'],
        })

    optimizer = optim.AdamW(param_groups)
    return optimizer


def train_one_epoch(model, loader, criterion, optimizer, device, mixup_alpha=0.0, mixup_prob=0.0):
    model.train()
    running_loss = 0.0
    running_correct = 0
    total = 0
    y_true = []
    y_pred = []

    pbar = tqdm(loader, desc='Train', leave=False)
    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        use_mixup = (mixup_alpha > 0) and (np.random.rand() < mixup_prob)
        if use_mixup:
            mixed_images, labels_a, labels_b, lam = _sample_mixup_data(images, labels, alpha=mixup_alpha)
            logits = model(mixed_images)
            loss = _mixup_loss(criterion, logits, labels_a, labels_b, lam)
        else:
            logits = model(images)
            loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        preds = torch.argmax(logits, dim=1)
        running_correct += (preds == labels).sum().item()
        total += batch_size

        y_true.extend(labels.detach().cpu().numpy().tolist())
        y_pred.extend(preds.detach().cpu().numpy().tolist())

    train_f1 = f1_score(y_true, y_pred, average='binary', zero_division=0)
    return {
        'loss': running_loss / max(total, 1),
        'acc': running_correct / max(total, 1),
        'f1': train_f1,
    }


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    running_correct = 0
    total = 0
    y_true = []
    y_pred = []

    pbar = tqdm(loader, desc='Eval', leave=False)
    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = model(images)
        loss = criterion(logits, labels)

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        preds = torch.argmax(logits, dim=1)
        running_correct += (preds == labels).sum().item()
        total += batch_size

        y_true.extend(labels.detach().cpu().numpy().tolist())
        y_pred.extend(preds.detach().cpu().numpy().tolist())

    val_f1 = f1_score(y_true, y_pred, average='binary', zero_division=0)
    return {
        'loss': running_loss / max(total, 1),
        'acc': running_correct / max(total, 1),
        'f1': val_f1,
    }


@torch.no_grad()
def collect_probs_and_labels(model, loader, device):
    model.eval()
    all_probs = []
    all_labels = []

    for images, labels in tqdm(loader, desc='Collect probs', leave=False):
        images = images.to(device, non_blocking=True)
        probs = torch.softmax(model(images), dim=1)[:, 1]
        all_probs.extend(probs.detach().cpu().numpy().tolist())
        all_labels.extend(labels.detach().cpu().numpy().tolist())

    return np.array(all_probs), np.array(all_labels)


@torch.no_grad()
def collect_test_prob_map(model, loader, device, use_tta=False, tta_mode='simple_flip', tta_views=4):
    model.eval()
    id_to_prob = {}

    for images, img_paths in tqdm(loader, desc='Collect test probs', leave=False):
        images = images.to(device, non_blocking=True)
        if use_tta:
            probs = _tta_forward_probs(model, images, mode=tta_mode, views=tta_views)
        else:
            probs = torch.softmax(model(images), dim=1)[:, 1]
        probs = probs.detach().cpu().numpy().tolist()
        for prob, path in zip(probs, img_paths):
            image_id = os.path.basename(path)
            id_to_prob[image_id] = float(prob)

    return id_to_prob


def optimize_threshold_bayes(y_true, y_prob, n_trials=40):
    def objective(trial):
        threshold = trial.suggest_float('threshold', 0.05, 0.95)
        y_hat = (y_prob >= threshold).astype(int)
        return f1_score(y_true, y_hat, average='binary', zero_division=0)

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return study.best_params['threshold'], study.best_value

In [ ]:
# Register/Login wandb
# Preferred: set WANDB_API_KEY in environment before running this cell.
wandb.login()

run = wandb.init(
    project=CFG['project_name'],
    name=CFG['run_name'],
    config=CFG,
)

clean_stats = {
    'clean/num_train_indices': int(len(train_indices)),
    'clean/num_relabels': int(len(train_relabel_map)),
    'clean/manual_file_used': int(manual_decision_df is not None),
}
if manual_decision_df is not None:
    for status_name in ['clean', 'hard', 'drop', 'relabel']:
        clean_stats[f'clean/status_{status_name}'] = int((manual_decision_df['sample_status'] == status_name).sum())
print('Cleaning stats:', clean_stats)
if wandb.run is not None:
    wandb.log(clean_stats)

In [ ]:
# Main training loop (two-stage finetuning + param-group LR)
history = {
    'train_loss': [],
    'train_acc': [],
    'train_f1': [],
    'val_loss': [],
    'val_acc': [],
    'val_f1': [],
    'lr_backbone': [],
    'lr_head': [],
}

best_val_loss = float('inf')
best_val_f1 = -1.0
best_ckpt_path = os.path.join(
    CFG['output_dir'],
    f"best_{CFG['model_name'].replace('/', '_').replace('.', '_')}.pt",
)

phase1_epochs = min(CFG['head_only_epochs'], CFG['num_epochs'])
phase2_epochs = max(0, CFG['num_epochs'] - phase1_epochs)
phases = [
    ('head_only', phase1_epochs, 0),
    ('finetune', phase2_epochs, CFG['unfreeze_last_n_blocks']),
]

global_epoch = 0
for phase_name, phase_epochs, unfreeze_n in phases:
    if phase_epochs <= 0:
        continue

    set_trainable_layers(model, unfreeze_last_n_blocks=unfreeze_n)
    optimizer = build_optimizer(model, CFG)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=phase_epochs)

    trainable_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_count = sum(p.numel() for p in model.parameters())
    print(
        f"Phase={phase_name} | epochs={phase_epochs} | trainable_params={trainable_count}/{total_count} "
        f"({100.0 * trainable_count / max(total_count, 1):.2f}%)"
    )

    for _ in range(phase_epochs):
        global_epoch += 1
        use_mixup = (phase_name == 'finetune') and (CFG['mixup_alpha'] > 0)
        train_metrics = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            device,
            mixup_alpha=CFG['mixup_alpha'] if use_mixup else 0.0,
            mixup_prob=CFG['mixup_prob'] if use_mixup else 0.0,
        )
        val_metrics = evaluate(model, val_loader, criterion, device)

        lr_backbone = optimizer.param_groups[0]['lr'] if len(optimizer.param_groups) > 1 else 0.0
        lr_head = optimizer.param_groups[-1]['lr']
        scheduler.step()

        history['train_loss'].append(train_metrics['loss'])
        history['train_acc'].append(train_metrics['acc'])
        history['train_f1'].append(train_metrics['f1'])
        history['val_loss'].append(val_metrics['loss'])
        history['val_acc'].append(val_metrics['acc'])
        history['val_f1'].append(val_metrics['f1'])
        history['lr_backbone'].append(lr_backbone)
        history['lr_head'].append(lr_head)

        if val_metrics['loss'] < best_val_loss:
            best_val_loss = val_metrics['loss']
            best_val_f1 = val_metrics['f1']
            torch.save(model.state_dict(), best_ckpt_path)

        if wandb.run is not None:
            wandb.log({
                'epoch': global_epoch,
                'phase': phase_name,
                'train/loss': train_metrics['loss'],
                'train/acc': train_metrics['acc'],
                'train/f1': train_metrics['f1'],
                'val/loss': val_metrics['loss'],
                'val/acc': val_metrics['acc'],
                'val/f1': val_metrics['f1'],
                'lr/backbone': lr_backbone,
                'lr/head': lr_head,
                'best_val_loss': best_val_loss,
                'best_val_f1_at_best_loss': best_val_f1,
            })

        print(
            f"Epoch [{global_epoch}/{CFG['num_epochs']}] phase={phase_name} "
            f"train_loss={train_metrics['loss']:.4f}, train_acc={train_metrics['acc']:.4f}, train_f1={train_metrics['f1']:.4f}, "
            f"val_loss={val_metrics['loss']:.4f}, val_acc={val_metrics['acc']:.4f}, val_f1={val_metrics['f1']:.4f}, "
            f"lr_backbone={lr_backbone:.2e}, lr_head={lr_head:.2e}"
        )

print('Best val loss:', f'{best_val_loss:.4f}')
print('Val f1 at best-loss checkpoint:', f'{best_val_f1:.4f}')
print('Best checkpoint:', best_ckpt_path)

In [ ]:
# Plot curves
epochs = np.arange(1, len(history['train_loss']) + 1)

plt.figure(figsize=(18, 4))

plt.subplot(1, 4, 1)
plt.plot(epochs, history['train_loss'], label='train_loss')
plt.plot(epochs, history['val_loss'], label='val_loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Loss')

plt.subplot(1, 4, 2)
plt.plot(epochs, history['train_acc'], label='train_acc')
plt.plot(epochs, history['val_acc'], label='val_acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Accuracy')

plt.subplot(1, 4, 3)
plt.plot(epochs, history['train_f1'], label='train_f1')
plt.plot(epochs, history['val_f1'], label='val_f1')
plt.xlabel('Epoch')
plt.ylabel('F1')
plt.legend()
plt.title('F1 Score')

plt.subplot(1, 4, 4)
plt.plot(epochs, history['lr_backbone'], label='lr_backbone')
plt.plot(epochs, history['lr_head'], label='lr_head')
plt.xlabel('Epoch')
plt.ylabel('Learning Rate')
plt.legend()
plt.title('Learning Rate')

plt.tight_layout()
plt.show()

In [ ]:
@torch.no_grad()
def predict(model, loader, device, threshold=0.5):
    model.eval()
    id_to_pred = {}

    for images, img_paths in tqdm(loader, desc='Predicting', leave=False):
        images = images.to(device, non_blocking=True)
        probs = torch.softmax(model(images), dim=1)[:, 1]
        preds = (probs >= threshold).long().cpu().numpy().tolist()

        for pred, path in zip(preds, img_paths):
            image_id = os.path.basename(path)
            id_to_pred[image_id] = int(pred)

    return id_to_pred


def make_submission(id_to_pred, template_csv_path, output_path):
    ids_df = pd.read_csv(template_csv_path)
    if 'id' not in ids_df.columns:
        raise ValueError(f"{template_csv_path} must contain 'id' column")

    submission_df = ids_df.copy()
    submission_df['label'] = submission_df['id'].map(id_to_pred)

    if submission_df['label'].isna().any():
        missing_ids = submission_df.loc[submission_df['label'].isna(), 'id'].head(5).tolist()
        raise RuntimeError(f"Missing predictions for some ids, examples: {missing_ids}")

    submission_df['label'] = submission_df['label'].astype(int)
    submission_df.to_csv(output_path, index=False)
    return submission_df

In [ ]:
# Load best checkpoint (optional but recommended before inference)
if os.path.exists(best_ckpt_path):
    model.load_state_dict(torch.load(best_ckpt_path, map_location=device))
    print('Loaded best checkpoint for inference.')
else:
    print('Best checkpoint not found, using current model weights.')

# Use Bayesian optimization to find a better decision threshold on validation F1
val_probs, val_labels = collect_probs_and_labels(model, val_loader, device)
best_threshold, best_f1_from_bayes = optimize_threshold_bayes(
    y_true=val_labels,
    y_prob=val_probs,
    n_trials=CFG['bayes_threshold_trials'],
)
print('Bayes-optimized threshold:', round(best_threshold, 4))
print('Best validation F1 from threshold search:', round(best_f1_from_bayes, 4))

# Prior-constrained threshold on test distribution
id_to_prob = collect_test_prob_map(
    model,
    test_loader,
    device,
    use_tta=CFG['use_tta'],
    tta_mode=CFG['tta_mode'],
    tta_views=CFG['tta_views'],
)
test_probs = np.array(list(id_to_prob.values()), dtype=np.float32)
target_pos_rate = float(CFG['test_class_prior'][1])
prior_threshold = float(np.quantile(test_probs, 1.0 - target_pos_rate))

if CFG['use_prior_threshold_on_test']:
    thresholds = np.arange(
        CFG['threshold_scan_min'],
        CFG['threshold_scan_max'] + 1e-12,
        CFG['threshold_scan_step'],
        dtype=np.float32,
    )
    f1_floor = float(best_f1_from_bayes - CFG['threshold_f1_tolerance'])
    pos_rate_min = float(CFG['test_pos_rate_min'])
    pos_rate_max = float(CFG['test_pos_rate_max'])
    pos_rate_target = float(CFG['test_pos_rate_target'])

    scan_records = []
    for threshold in thresholds:
        val_pred = (val_probs >= threshold).astype(np.int64)
        val_f1 = float(f1_score(val_labels, val_pred))
        test_pos_rate = float(np.mean(test_probs >= threshold))
        pass_rate = (pos_rate_min <= test_pos_rate <= pos_rate_max)
        pass_f1 = (val_f1 >= f1_floor)
        scan_records.append({
            'threshold': float(threshold),
            'val_f1': val_f1,
            'test_pos_rate': test_pos_rate,
            'rate_distance': abs(test_pos_rate - pos_rate_target),
            'pass_rate': bool(pass_rate),
            'pass_f1': bool(pass_f1),
        })

    scan_df = pd.DataFrame(scan_records)
    scan_csv_path = os.path.join(CFG['output_dir'], 'threshold_scan_metrics.csv')
    scan_df.to_csv(scan_csv_path, index=False)

    feasible_df = scan_df[scan_df['pass_rate'] & scan_df['pass_f1']].copy()
    if len(feasible_df) > 0:
        feasible_df = feasible_df.sort_values(
            by=['val_f1', 'rate_distance', 'threshold'],
            ascending=[False, True, True],
        )
        selected_row = feasible_df.iloc[0]
        selection_mode = 'feasible'
    else:
        f1_shortfall = np.maximum(f1_floor - scan_df['val_f1'].to_numpy(dtype=np.float64), 0.0)
        test_rate_values = scan_df['test_pos_rate'].to_numpy(dtype=np.float64)
        rate_penalty = np.where(
            test_rate_values < pos_rate_min,
            pos_rate_min - test_rate_values,
            np.where(test_rate_values > pos_rate_max, test_rate_values - pos_rate_max, 0.0),
        )
        scan_df['selection_penalty'] = (
            CFG['threshold_fallback_f1_penalty_weight'] * f1_shortfall + rate_penalty
        )
        fallback_df = scan_df.sort_values(
            by=['selection_penalty', 'rate_distance', 'threshold'],
            ascending=[True, True, True],
        )
        selected_row = fallback_df.iloc[0]
        selection_mode = 'fallback'
        print('Warning: no threshold satisfies both F1 floor and test positive-rate band; using fallback.')

    final_threshold = float(selected_row['threshold'])
    print('Threshold selection mode:', selection_mode)
    print('Threshold scan saved to:', scan_csv_path)
    print('F1 floor:', round(f1_floor, 4), 'band:', (pos_rate_min, pos_rate_max), 'target:', pos_rate_target)
    print('Selected row:', {
        'threshold': round(float(selected_row['threshold']), 4),
        'val_f1': round(float(selected_row['val_f1']), 4),
        'test_pos_rate': round(float(selected_row['test_pos_rate']), 4),
    })
else:
    final_threshold = float(best_threshold)
    print('Constrained threshold search disabled; using best_threshold only.')

id_to_pred = {img_id: int(prob >= final_threshold) for img_id, prob in id_to_prob.items()}
pred_pos_rate = float(np.mean(list(id_to_pred.values())))

print('TTA enabled:', CFG['use_tta'])
print('TTA mode/views:', CFG['tta_mode'], CFG['tta_views'])
print('Prior threshold:', round(prior_threshold, 4))
print('Final threshold used:', round(final_threshold, 4))
print('Predicted positive rate on test:', round(pred_pos_rate, 4))

# Export submission.csv
_, template_csv_path = resolve_test_paths(DATA_ROOT)
submission_path = os.path.join(CFG['output_dir'], 'submission.csv')
submission_df = make_submission(id_to_pred, template_csv_path, submission_path)

print('Submission saved to:', submission_path)
print('Rows in submission:', len(submission_df))
display(submission_df.head())

In [ ]:
# Final metrics + end-of-training error analysis
print('best_val_loss =', round(best_val_loss, 4))
print('best_val_f1_at_best_loss =', round(best_val_f1, 4))
print('best_threshold =', round(best_threshold, 4))
print('best_f1_from_bayes =', round(best_f1_from_bayes, 4))
print('tta_enabled =', CFG['use_tta'], 'tta_mode =', CFG['tta_mode'], 'tta_views =', CFG['tta_views'])
print('last_train_acc =', round(history['train_acc'][-1], 4), 'last_val_acc =', round(history['val_acc'][-1], 4))
print('last_train_f1 =', round(history['train_f1'][-1], 4), 'last_val_f1 =', round(history['val_f1'][-1], 4))

# Validation predictions with optimized threshold
val_pred = (val_probs >= best_threshold).astype(int)
val_acc = accuracy_score(val_labels, val_pred)
val_prec = precision_score(val_labels, val_pred, zero_division=0)
val_rec = recall_score(val_labels, val_pred, zero_division=0)
val_f1_opt = f1_score(val_labels, val_pred, zero_division=0)
print('Optimized-threshold val metrics:', {
    'acc': round(val_acc, 4),
    'precision': round(val_prec, 4),
    'recall': round(val_rec, 4),
    'f1': round(val_f1_opt, 4),
})

# Confusion matrix
cm = confusion_matrix(val_labels, val_pred, labels=[0, 1])
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['pred_non_meteorite', 'pred_meteorite'])
ax.set_yticklabels(['true_non_meteorite', 'true_meteorite'])
ax.set_xlabel('Prediction')
ax.set_ylabel('Ground Truth')
ax.set_title('Validation Confusion Matrix')
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', color='black', fontsize=12)
plt.colorbar(im, ax=ax)
plt.tight_layout()
cm_path = os.path.join(CFG['output_dir'], 'val_confusion_matrix.png')
plt.savefig(cm_path, dpi=140)
plt.show()

# Build per-sample error table
val_paths = [val_dataset_raw.samples[i] for i in val_indices]
val_error_df = pd.DataFrame({
    'path': val_paths,
    'id': [os.path.basename(p) for p in val_paths],
    'y_true': val_labels.astype(int),
    'y_prob_meteorite': val_probs.astype(float),
    'y_pred': val_pred.astype(int),
})
val_error_df['is_error'] = val_error_df['y_true'] != val_error_df['y_pred']
val_error_df['error_type'] = np.where(
    val_error_df['is_error'] & (val_error_df['y_true'] == 1),
    'FN_meteorite_missed',
    np.where(
        val_error_df['is_error'] & (val_error_df['y_true'] == 0),
        'FP_false_alarm',
        'correct',
    ),
)
val_error_df['error_confidence'] = np.where(
    val_error_df['y_pred'] == 1,
    val_error_df['y_prob_meteorite'],
    1.0 - val_error_df['y_prob_meteorite'],
)

error_csv_path = os.path.join(CFG['output_dir'], 'val_error_analysis.csv')
val_error_df.sort_values(['is_error', 'error_confidence'], ascending=[False, False]).to_csv(error_csv_path, index=False)
print('Saved error analysis CSV:', error_csv_path)
print('Total validation errors:', int(val_error_df['is_error'].sum()))
display(val_error_df[val_error_df['is_error']].head(15))

# Log to wandb once at training end
if wandb.run is not None:
    wandb.log({
        'val/opt_threshold_acc': val_acc,
        'val/opt_threshold_precision': val_prec,
        'val/opt_threshold_recall': val_rec,
        'val/opt_threshold_f1': val_f1_opt,
        'val/opt_threshold': best_threshold,
        'val/confusion_matrix': wandb.Image(cm_path),
    })
    error_examples = val_error_df[val_error_df['is_error']].head(24)
    images_to_log = []
    for _, row in error_examples.iterrows():
        caption = f"id={row['id']} true={row['y_true']} pred={row['y_pred']} prob={row['y_prob_meteorite']:.3f} type={row['error_type']}"
        images_to_log.append(wandb.Image(row['path'], caption=caption))
    if images_to_log:
        wandb.log({'val/error_examples': images_to_log})

# Finalize wandb run
if wandb.run is not None:
    wandb.finish()